In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/waleedejaz/seoul-bike-sharing-demand/SeoulBikeData.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

In [3]:
df = pd.read_csv('/kaggle/input/datasets/waleedejaz/seoul-bike-sharing-demand/SeoulBikeData.csv', encoding='unicode_escape')

In [4]:
df.dropna(axis=0, subset=['Rented Bike Count'], inplace=True)
df.drop(['Date'], axis=1, inplace=True)

In [5]:
features = df.columns[1:]
X = df[features]
y = df['Rented Bike Count']
features
X

,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
8755,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes


In [6]:
# I wanted to experiment on the model complexity without too much Feature Engineering, so not much preprocessing initially
# I have just one hot encoded the categorial features for Linear Regression Baseline

X = pd.get_dummies(X, columns=['Hour', 'Seasons', 'Holiday', 'Functioning Day'], drop_first=True, dtype=int)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

print("LINEAR REGRESSION:\n")

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

print("\nDECISION TREE REGRESSOR:\n")

model = DecisionTreeRegressor(max_depth=18, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

print("\nXGBOOST REGRESSOR:\n")
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.1,
    early_stopping_rounds=50,
    n_jobs=4,
    seed=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

# NOTE: Results are clearly demonstrating the effect of model complexity which I wanted to see XGBoost miles better than Linear Regression
# and surprisingly much much better than a DTRegressor
# GREAT START!

LINEAR REGRESSION:

RMSE: 374.4176
MAE: 281.8237

RMSE: 376.5360
MAE: 285.8975

DECISION TREE REGRESSOR:

RMSE: 288.5615
MAE: 181.2777

RMSE: 295.6182
MAE: 183.0501

XGBOOST REGRESSOR:

RMSE: 232.9298
MAE: 149.5629

RMSE: 244.4034
MAE: 155.7212


In [8]:
n = len(X)

# NOTE: I am splitting using plain slicing since I have manually verified the dataframe is sorted day wise and for same day hour wise
# NOTE: Not using train test split since this is time based data and hence training on 8 am and 10 am and predicting for 9 am is unfair and hence using 
# a chronological split as I have also highlighted in the course proposal. 

train_end = int(0.6 * n)
val_end = int(0.8 * n)

X_train = X.iloc[:train_end]
X_val   = X.iloc[train_end:val_end]
X_test  = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val   = y.iloc[train_end:val_end]
y_test  = y.iloc[val_end:]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

RMSE: 607.0256
MAE: 513.4486

RMSE: 369.8886
MAE: 277.0216


In [9]:
model = DecisionTreeRegressor(max_depth=18, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

RMSE: 472.4594
MAE: 342.2714

RMSE: 517.6507
MAE: 348.0296


In [10]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score 

model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.1,
    early_stopping_rounds=50,
    n_jobs=4,
    seed=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')



RMSE: 483.4648
MAE: 395.9133

RMSE: 499.4878
MAE: 377.6833


## WHAT THE HELL HAPPENED HERE ?? COMPLETELY CONTRASTING RESULTS 😮😭😭😭

In [11]:
# So, I thought about this for a while, I think on chronological split it is happening
# that the training is happening almost exclusively in Winter months, 
# validation on summer months and testing on Autumn months. This could be the main problem I feel
# but how this helps linear regression generalise and not the other complex models, I have NO IDEA. 
# Have to investigate

# This is incredible man, wow, completely caught me off guards but now time to investigate. 

for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print(target.describe())

# NOTE: yeah so this shows the distribution changes (mean and std vary across the splits)


Train
count    5256.000000
mean      600.921613
std       631.756658
min         0.000000
25%       159.000000
50%       334.000000
75%       913.000000
max      3556.000000
Name: Rented Bike Count, dtype: float64

Validation
count    1752.000000
mean      963.248288
std       636.327062
min         0.000000
25%       498.000000
50%       827.000000
75%      1342.500000
max      3298.000000
Name: Rented Bike Count, dtype: float64

Test
count    1752.000000
mean      756.997146
std       613.507451
min         0.000000
25%       191.000000
50%       706.500000
75%      1138.750000
max      3154.000000
Name: Rented Bike Count, dtype: float64


In [12]:
'''
This result just doesn't sit right with me man, because to be honest this goes against what I would
have thought would happen based on theory.
I am not saying this just because the numbers align with my expectations, I swear no. But what is 
happening on the chronological split is a model is shown one end of a distribution and is judged on a 
completely unseen distribution. In fact that was the exact reason we had a random train test split and 
not a slicing. 
I know the reasoning for a chronological split - the prediction we are making is of future using past 
data so the model should be judged on that action... that is the reasoning for time based datasets. 
BUT come to think of it, seasons, temperature, humidity all play A MUCH LARGER role in bike demand and 
whether it is 2017 or 2019 really doesn't impact much for this particular problem if you think 
carefully. For general temporal data sure, but for this particular problem, I don't see why a random 
split is not the better option
'''

"\nThis result just doesn't sit right with me man, because to be honest this goes against what I would\nhave thought would happen based on theory.\nI am not saying this just because the numbers align with my expectations, I swear no. But what is \nhappening on the chronological split is a model is shown one end of a distribution and is judged on a \ncompletely unseen distribution. In fact that was the exact reason we had a random train test split and \nnot a slicing. \nI know the reasoning for a chronological split - the prediction we are making is of future using past \ndata so the model should be judged on that action... that is the reasoning for time based datasets. \nBUT come to think of it, seasons, temperature, humidity all play A MUCH LARGER role in bike demand and \nwhether it is 2017 or 2019 really doesn't impact much for this particular problem if you think \ncarefully. For general temporal data sure, but for this particular problem, I don't see why a random \nsplit is not th

## A NEW SPLIT TECHNIQUE MIXING BEST OF BOTH WORLDS

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/waleedejaz/seoul-bike-sharing-demand/SeoulBikeData.csv', encoding='unicode-escape')

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

days = df[["Date", "Seasons"]].drop_duplicates().reset_index(drop=True)

print(days.shape)
print(days["Seasons"].value_counts())

(365, 2)
Seasons
Spring    92
Summer    92
Autumn    91
Winter    90
Name: count, dtype: int64


In [16]:
train_days, remaining_days = train_test_split(
    days,
    test_size=0.4,
    random_state=42,
    stratify=days["Seasons"]
)

val_days, test_days = train_test_split(
    remaining_days,
    test_size=0.5,
    random_state=42,
    stratify=remaining_days["Seasons"]
)

train_mask = df["Date"].isin(train_days["Date"])
val_mask = df["Date"].isin(val_days["Date"])
test_mask = df["Date"].isin(test_days["Date"])

X = df.drop(columns=["Date", "Rented Bike Count"])

X = pd.get_dummies(
    X,
    columns=["Hour", "Seasons", "Holiday", "Functioning Day"],
    drop_first=True,
    dtype=int
)

y = df["Rented Bike Count"]

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]


In [17]:

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nSeason distribution:")
for name, mask in [
    ("Train", train_mask),
    ("Validation", val_mask),
    ("Test", test_mask)
]:
    print(name)
    print(df.loc[mask, "Seasons"].value_counts(normalize=True))

Train: (5256, 36) (5256,)
Validation: (1752, 36) (1752,)
Test: (1752, 36) (1752,)

Season distribution:
Train
Seasons
Spring    0.251142
Summer    0.251142
Autumn    0.251142
Winter    0.246575
Name: proportion, dtype: float64
Validation
Seasons
Summer    0.260274
Winter    0.246575
Spring    0.246575
Autumn    0.246575
Name: proportion, dtype: float64
Test
Seasons
Spring    0.260274
Winter    0.246575
Summer    0.246575
Autumn    0.246575
Name: proportion, dtype: float64


In [18]:
assert set(train_days["Date"]).isdisjoint(set(val_days["Date"]))
assert set(train_days["Date"]).isdisjoint(set(test_days["Date"]))
assert set(val_days["Date"]).isdisjoint(set(test_days["Date"]))

In [19]:
print("LINEAR REGRESSION:\n")

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

print("\nDECISION TREE REGRESSOR:\n")

model = DecisionTreeRegressor(max_depth=18, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')

print("\nXGBOOST REGRESSOR:\n")
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.1,
    early_stopping_rounds=50,
    n_jobs=4,
    seed=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred = model.predict(X_test)
y_pred_val = model.predict(X_val)

print(f'RMSE: {root_mean_squared_error(y_pred_val, y_val):.4f}')
print(f'MAE: {mean_absolute_error(y_pred_val, y_val):.4f}')
print()
print(f'RMSE: {root_mean_squared_error(y_pred, y_test):.4f}')
print(f'MAE: {mean_absolute_error(y_pred, y_test):.4f}')


LINEAR REGRESSION:

RMSE: 362.9971
MAE: 269.6009

RMSE: 383.6370
MAE: 293.1267

DECISION TREE REGRESSOR:

RMSE: 311.2711
MAE: 203.6951

RMSE: 290.6824
MAE: 186.1559

XGBOOST REGRESSOR:

RMSE: 258.5067
MAE: 164.8088

RMSE: 250.4534
MAE: 166.0807
